# Parkinson's Disease Voice Screening & Clinical Decision Support Platform
## Notebook 08: 10-Second Multi-Task Windowing, Subject-Level Splitting & WavLM Feature Extraction

> **CLINICAL RESEARCH NOTICE:** This platform is a clinical decision support and risk stratification research tool, **NOT** an autonomous diagnostic device. It does not replace comprehensive neurological evaluation, UPDRS scoring, or clinical examination.

---

### Objectives of this Notebook:
1. **10-Second Acoustic Windowing:** Upgrade from short 4.0s clips to **10.0-second temporal windows** (160,000 samples @ 16 kHz) to capture continuous dysarthric prosody, sentence-level speech rhythm, phonatory tremor, and articulatory decay.
2. **Multi-Dataset Curation:**
   - **IPVS (Italian Parkinson's Voice and Speech):** Sustained vowels (/a/, /e/, /i/, /o/, /u/), syllable DDK (/pa-ta-ka/), and continuous passage reading ("Il brano").
   - **MDVR-KCL (Zenodo 2867216):** Smartphone recordings from 37 speakers (21 healthy controls, 16 Parkinson's) reading continuous text passages (~150s each). *Note: Spontaneous dialogue is excluded to eliminate clinical interviewer speech confounders.*
3. **Strict Subject-Level Stratification:** Zero subject overlap across Train, Validation, and Test splits.
4. **WavLM-Base-Plus Feature Caching:** Extract and cache frozen WavLM representations into FP16 `.npy` files of fixed dimension $(T=499, D=768)$ with full Drive-backed resumability.

In [ ]:
# Cell 1: Environment Setup, Dependencies & Hardware Verification
import sys
import os
import subprocess
from pathlib import Path

# Detect Google Colab environment
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Running in Google Colab environment.")
    # Install required packages
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers", "torch", "torchaudio", "soundfile", "librosa", "pandas", "numpy", "tqdm", "scikit-learn"
    ])
    # Mount Google Drive for persistent caching
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/parkinsons_platform")
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
else:
    print("Running in local development environment.")
    cwd = Path.cwd()
    if cwd.name == "notebooks":
        PROJECT_ROOT = cwd.parent.parent
    else:
        PROJECT_ROOT = cwd

print(f"Project Root Directory: {PROJECT_ROOT}")

# Hardware acceleration verification
import torch
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA GPU Active: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Apple Silicon MPS Active.")
else:
    device = torch.device("cpu")
    print("Using CPU Compute Device.")

In [ ]:
# Cell 2: Directory Resolution & Raw Dataset Verification
import os
import json
import pandas as pd
from huggingface_hub import snapshot_download

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

IPVS_DIR = RAW_DIR / "ipvs"
MDVR_DIR = RAW_DIR / "mdvr_kcl"
AUDIO_10S_DIR = PROCESSED_DIR / "audio_10s"
FEATURES_10S_DIR = PROCESSED_DIR / "features_10s"

for d in [AUDIO_10S_DIR, FEATURES_10S_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 1. Verify / Acquire IPVS Dataset
if not IPVS_DIR.exists() or len(list(IPVS_DIR.rglob("*.wav"))) == 0:
    print("Downloading IPVS dataset from Hugging Face mirror (birgermoell/Italian_Parkinsons_Voice_and_Speech)...")
    IPVS_DIR.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id="birgermoell/Italian_Parkinsons_Voice_and_Speech",
        repo_type="dataset",
        local_dir=str(IPVS_DIR),
        local_dir_use_symlinks=False,
        resume_download=True
    )

ipvs_wav_count = len(list(IPVS_DIR.rglob("*.wav")))
print(f"IPVS Dataset verified: {ipvs_wav_count} audio files in {IPVS_DIR}")

# 2. Verify MDVR-KCL Dataset
# ReadText directory contains HC/ and PD/ paragraph readings
MDVR_READ_TEXT_DIR = MDVR_DIR / "extracted" / "26-29_09_2017_KCL" / "ReadText"
if not MDVR_READ_TEXT_DIR.exists():
    alt_dir = MDVR_DIR / "ReadText"
    if alt_dir.exists():
        MDVR_READ_TEXT_DIR = alt_dir

assert MDVR_READ_TEXT_DIR.exists(), (
    f"MDVR ReadText directory missing at {MDVR_READ_TEXT_DIR}. "
    "Please ensure MDVR-KCL ReadText files are unzipped into data/raw/mdvr_kcl/extracted/26-29_09_2017_KCL/ReadText/"
)

mdvr_wav_count = len(list(MDVR_READ_TEXT_DIR.glob("*/*.wav")))
print(f"MDVR-KCL ReadText verified: {mdvr_wav_count} audio files across HC and PD cohorts.")

In [ ]:
# Cell 3: Subject-Level Stratification & Strict Leakage Audit
# Load existing IPVS split manifest to maintain continuity
IPVS_MANIFEST_PATH = PROCESSED_DIR / "split_manifest.json"
assert IPVS_MANIFEST_PATH.exists(), f"Split manifest missing at {IPVS_MANIFEST_PATH}"

with open(IPVS_MANIFEST_PATH, "r", encoding="utf-8") as f:
    ipvs_manifest = json.load(f)

ipvs_split_map = {}
for split_name in ["train", "val", "test"]:
    for sid in ipvs_manifest[split_name]:
        ipvs_split_map[sid] = split_name

# MDVR-KCL Subject Splitting (37 subjects: 21 HC, 16 PD, Seed 42 Stratified)
# Excludes SpontaneousDialogue to eliminate physician vocal confounders
MDVR_SPLITS = {
    "train": [
        # 15 Healthy Control
        "ID00", "ID28", "ID25", "ID01", "ID12", "ID09", "ID19", "ID05", "ID31", "ID26", "ID22", "ID03", "ID14", "ID36", "ID08",
        # 11 Parkinson's Disease
        "ID18", "ID32", "ID27", "ID07", "ID34", "ID17", "ID06", "ID30", "ID20", "ID16", "ID24"
    ],
    "val": [
        # 3 Healthy Control
        "ID21", "ID11", "ID15",
        # 2 Parkinson's Disease
        "ID33", "ID02"
    ],
    "test": [
        # 3 Healthy Control
        "ID23", "ID35", "ID10",
        # 3 Parkinson's Disease
        "ID13", "ID29", "ID04"
    ]
}

mdvr_split_map = {}
for split_name, sids in MDVR_SPLITS.items():
    for sid in sids:
        mdvr_split_map[sid] = split_name

# Aggregate unique subjects across both datasets
train_subjects = set(ipvs_manifest["train"]) | set(MDVR_SPLITS["train"])
val_subjects = set(ipvs_manifest["val"]) | set(MDVR_SPLITS["val"])
test_subjects = set(ipvs_manifest["test"]) | set(MDVR_SPLITS["test"])

print("=" * 60)
print("SUBJECT-LEVEL STRATIFICATION AUDIT")
print("=" * 60)
print(f"  Train Subjects:       {len(train_subjects):3d} (IPVS: {len(ipvs_manifest['train'])}, MDVR: {len(MDVR_SPLITS['train'])})")
print(f"  Validation Subjects:  {len(val_subjects):3d} (IPVS: {len(ipvs_manifest['val'])}, MDVR: {len(MDVR_SPLITS['val'])})")
print(f"  Test Subjects:        {len(test_subjects):3d} (IPVS: {len(ipvs_manifest['test'])}, MDVR: {len(MDVR_SPLITS['test'])})")
print(f"  Total Unique Subjects:{len(train_subjects | val_subjects | test_subjects):3d}")

# STRICT DATA CONTRACT LEAKAGE AUDIT ASSERTIONS
assert len(train_subjects & val_subjects) == 0, f"Leakage between Train & Val: {train_subjects & val_subjects}"
assert len(train_subjects & test_subjects) == 0, f"Leakage between Train & Test: {train_subjects & test_subjects}"
assert len(val_subjects & test_subjects) == 0, f"Leakage between Val & Test: {val_subjects & test_subjects}"

print("\n" + "=" * 60)
print("PASS: Zero subject leakage across all splits.")
print("=" * 60)

In [ ]:
# Cell 4: 10-Second Window Extraction Pipeline
# Windowing rules:
#   - trimmed >= 10.0s: sliding windows of 10.0s with 5.0s stride (padded_fraction = 0.0)
#   - 8.0s <= trimmed < 10.0s: repeat-pad up to 10.0s (padded_fraction <= 20%)
#   - trimmed < 8.0s: excluded and logged
import soundfile as sf
import numpy as np
import librosa
from dataclasses import dataclass
from typing import List, Tuple, Dict
from tqdm import tqdm

@dataclass
class WindowConfig:
    target_sr: int = 16000
    trim_top_db: int = 30
    normalize: bool = True
    segment_seconds: float = 10.0
    window_stride: float = 5.0
    min_valid_seconds: float = 8.0
    max_pad_fraction: float = 0.20
    pad_mode: str = "repeat"

    @property
    def target_samples(self) -> int:
        return int(round(self.segment_seconds * self.target_sr))

def extract_windows_from_waveform(waveform: np.ndarray, sample_rate: int, config: WindowConfig) -> List[Tuple[np.ndarray, float, float]]:
    # Extract 10s windows with sliding and repeat-padding
    if waveform is None or waveform.size == 0:
        return []

    # 1. Convert to float32
    if np.issubdtype(waveform.dtype, np.integer):
        max_val = float(np.iinfo(waveform.dtype).max)
        waveform = waveform.astype(np.float32) / max_val
    else:
        waveform = waveform.astype(np.float32)

    # 2. Convert to mono
    if waveform.ndim > 1:
        waveform = np.mean(waveform, axis=0) if waveform.shape[0] < waveform.shape[1] else np.mean(waveform, axis=-1)
    waveform = waveform.flatten()

    # 3. Resample to 16 kHz
    if sample_rate != config.target_sr:
        waveform = librosa.resample(waveform, orig_sr=sample_rate, target_sr=config.target_sr, res_type="soxr_hq")

    # 4. Trim leading/trailing silence
    trimmed, _ = librosa.effects.trim(waveform, top_db=config.trim_top_db)
    if len(trimmed) == 0:
        return []

    # 5. Peak normalization
    if config.normalize:
        peak = np.max(np.abs(trimmed))
        if peak > 1e-8:
            trimmed = trimmed / peak

    trimmed_dur = len(trimmed) / config.target_sr
    target_samples = config.target_samples

    # Filter out files shorter than 8.0s
    if trimmed_dur < config.min_valid_seconds:
        return []

    # Repeat-pad short valid files (8.0s - 10.0s)
    if trimmed_dur < config.segment_seconds:
        pad_fraction = float((config.segment_seconds - trimmed_dur) / config.segment_seconds)
        repeats = int(np.ceil(target_samples / max(len(trimmed), 1)))
        window = np.tile(trimmed, repeats)[:target_samples]
        return [(np.ascontiguousarray(window, dtype=np.float32), 0.0, pad_fraction)]

    # Sliding window for >= 10.0s
    stride_samples = int(round(config.window_stride * config.target_sr))
    windows = []
    start_sample = 0
    while start_sample + target_samples <= len(trimmed):
        win = trimmed[start_sample : start_sample + target_samples]
        start_sec = float(start_sample / config.target_sr)
        windows.append((np.ascontiguousarray(win, dtype=np.float32), start_sec, 0.0))
        start_sample += stride_samples

    return windows

win_config = WindowConfig()
print("WindowConfig initialized: 10.0s windows, 5.0s stride, 8.0s minimum, repeat-padding <= 20%.")

In [ ]:
# Cell 5: Process Audio Files and Generate metadata_10s.csv (Resumable)
# Discover IPVS files
ipvs_records = []
for wav_path in sorted(list(IPVS_DIR.rglob("*.wav"))):
    parts = wav_path.parts
    try:
        group_idx = next(i for i, p in enumerate(parts) if any(k in p for k in ["Young", "Elderly", "Parkinson"]))
    except StopIteration:
        continue
    group_str = parts[group_idx]
    fname = wav_path.name
    if "Young" in group_str:
        label = 0
        sub_name = parts[group_idx + 1].strip().replace(" ", "_")
        subject_id = f"yhc_{sub_name}"
    elif "Elderly" in group_str:
        label = 0
        sub_name = parts[group_idx + 1].strip().replace(" ", "_")
        subject_id = f"ehc_{sub_name}"
    elif "Parkinson" in group_str:
        label = 1
        batch = parts[group_idx + 1].strip().replace(" ", "_")
        sub_name = parts[group_idx + 2].strip().replace(" ", "_")
        subject_id = f"pd_{batch}_{sub_name}"
    else:
        continue

    fname_upper = fname.upper()
    if fname_upper.startswith(("VA", "VE", "VI", "VO", "VU")):
        task_type = "vowel_sustain"
    elif fname_upper.startswith("PR"):
        task_type = "reading_passage"
    elif fname_upper.startswith(("B1", "B2", "D1", "D2", "FB1")):
        task_type = "syllable_repetition"
    else:
        task_type = "syllable_repetition"

    if subject_id in ipvs_split_map:
        ipvs_records.append({
            "source_path": wav_path,
            "subject_id": subject_id,
            "dataset": "ipvs",
            "task_type": task_type,
            "label": label,
            "split": ipvs_split_map[subject_id],
            "file_stem": wav_path.stem
        })

# Discover MDVR ReadText files
mdvr_records = []
for cls_folder, label in [("HC", 0), ("PD", 1)]:
    cls_dir = MDVR_READ_TEXT_DIR / cls_folder
    if not cls_dir.exists():
        continue
    for wav_path in sorted(list(cls_dir.glob("*.wav"))):
        sid = wav_path.stem.split("_")[0]
        if sid in mdvr_split_map:
            mdvr_records.append({
                "source_path": wav_path,
                "subject_id": sid,
                "dataset": "mdvr_kcl",
                "task_type": "reading_passage",
                "label": label,
                "split": mdvr_split_map[sid],
                "file_stem": wav_path.stem
            })

all_recordings = ipvs_records + mdvr_records
print(f"Total recordings discovered: {len(all_recordings)} (IPVS: {len(ipvs_records)}, MDVR: {len(mdvr_records)})")

# Execute Window Extraction & Save Audio
metadata_rows = []
excluded_files = 0

for rec in tqdm(all_recordings, desc="Extracting 10s Windows"):
    try:
        audio, sr = sf.read(str(rec["source_path"]))
    except Exception as exc:
        print(f"Error reading {rec['source_path']}: {exc}")
        continue

    windows = extract_windows_from_waveform(audio, sr, win_config)
    if not windows:
        excluded_files += 1
        continue

    subj_audio_dir = AUDIO_10S_DIR / rec["split"] / rec["subject_id"]
    subj_feat_dir = FEATURES_10S_DIR / rec["split"] / rec["subject_id"]
    subj_audio_dir.mkdir(parents=True, exist_ok=True)
    subj_feat_dir.mkdir(parents=True, exist_ok=True)

    for idx, (win_arr, start_sec, pad_frac) in enumerate(windows):
        win_fname = f"{rec['file_stem']}_w{idx:03d}_{start_sec:.1f}s.wav"
        feat_fname = f"{rec['file_stem']}_w{idx:03d}_{start_sec:.1f}s.npy"

        win_path = subj_audio_dir / win_fname
        feat_path = subj_feat_dir / feat_fname

        # Save audio window if not already present
        if not win_path.exists():
            sf.write(str(win_path), win_arr, win_config.target_sr, subtype="PCM_16")

        metadata_rows.append({
            "window_path": win_path.relative_to(PROJECT_ROOT).as_posix(),
            "feature_path": feat_path.relative_to(PROJECT_ROOT).as_posix(),
            "subject_id": rec["subject_id"],
            "dataset": rec["dataset"],
            "task_type": rec["task_type"],
            "label": rec["label"],
            "split": rec["split"],
            "window_start_sec": round(float(start_sec), 2),
            "padded_fraction": round(float(pad_frac), 4)
        })

df_meta = pd.DataFrame(metadata_rows)
METADATA_10S_PATH = PROCESSED_DIR / "metadata_10s.csv"
df_meta.to_csv(METADATA_10S_PATH, index=False)
print(f"Saved metadata: {METADATA_10S_PATH} with {len(df_meta)} rows.")
print(f"Excluded {excluded_files} recordings shorter than 8.0s.")
print(df_meta.head(5))

In [ ]:
# Cell 6: Frozen WavLM-Base-Plus Feature Extraction & FP16 Caching
# Sequence Length Contract: 10.0s (160,000 samples @ 16kHz) -> T = 499 frames, D = 768 features
from transformers import WavLMModel

MODEL_ID = "microsoft/wavlm-base-plus"
print(f"Loading foundation model: {MODEL_ID}...")
wavlm_model = WavLMModel.from_pretrained(MODEL_ID)
wavlm_model.to(device)
wavlm_model.eval()
wavlm_model.requires_grad_(False)

# Verify zero trainable parameters
unfrozen = [n for n, p in wavlm_model.named_parameters() if p.requires_grad]
assert len(unfrozen) == 0, f"Found unfrozen parameters: {unfrozen}"
print("Verified: WavLM is completely frozen (0 trainable parameters).")

# Verify empirical sequence length T for 160,000 samples
dummy_input = torch.zeros(1, win_config.target_samples, device=device)
with torch.no_grad():
    if device.type == "cuda":
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            dummy_out = wavlm_model(dummy_input).last_hidden_state
    else:
        dummy_out = wavlm_model(dummy_input).last_hidden_state

EXACT_T, EXACT_D = dummy_out.shape[1], dummy_out.shape[2]
print(f"Fixed WavLM Sequence Length Contract: T = {EXACT_T}, Feature Dim: D = {EXACT_D}")
assert EXACT_T == 499, f"Expected T=499, got {EXACT_T}"
assert EXACT_D == 768, f"Expected D=768, got {EXACT_D}"

# Resumable feature extraction loop
BATCH_SIZE = 32 if device.type == "cuda" else 8
use_amp = (device.type == "cuda")

# Find uncached windows
pending_indices = []
for idx, row in df_meta.iterrows():
    feat_path = PROJECT_ROOT / row["feature_path"]
    if not feat_path.exists():
        pending_indices.append(idx)

print(f"Total Windows: {len(df_meta)} | Already Cached: {len(df_meta) - len(pending_indices)} | Pending Extraction: {len(pending_indices)}")

if pending_indices:
    for i in tqdm(range(0, len(pending_indices), BATCH_SIZE), desc="Caching WavLM Features"):
        batch_slice = pending_indices[i : i + BATCH_SIZE]
        batch_tensors = []
        batch_dest_paths = []

        for b_idx in batch_slice:
            row = df_meta.iloc[b_idx]
            w_path = PROJECT_ROOT / row["window_path"]
            f_path = PROJECT_ROOT / row["feature_path"]
            arr, _ = sf.read(str(w_path))
            batch_tensors.append(torch.from_numpy(arr).float())
            batch_dest_paths.append(f_path)

        batch_audio = torch.stack(batch_tensors, dim=0).to(device)

        with torch.no_grad():
            if use_amp:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    feats = wavlm_model(batch_audio).last_hidden_state
            else:
                feats = wavlm_model(batch_audio).last_hidden_state

        feats_np = feats.detach().cpu().to(torch.float16).numpy()

        for feat_arr, f_path in zip(feats_np, batch_dest_paths):
            f_path.parent.mkdir(parents=True, exist_ok=True)
            np.save(str(f_path), feat_arr)

print("All WavLM feature tensors cached successfully.")

In [ ]:
# Cell 7: Full Integrity Verification & Quality Audit Checklist
print("=== METADATA & FEATURE CACHE INTEGRITY AUDIT ===")

# 1. Metadata file existence & column contract
assert METADATA_10S_PATH.exists(), f"Missing metadata file: {METADATA_10S_PATH}"
df_audit = pd.read_csv(METADATA_10S_PATH)

REQUIRED_COLUMNS = [
    "window_path", "feature_path", "subject_id", "dataset",
    "task_type", "label", "split", "window_start_sec", "padded_fraction"
]
assert list(df_audit.columns) == REQUIRED_COLUMNS, f"Column mismatch! Got {list(df_audit.columns)}"
print("Strict metadata schema verified: all 9 required columns match contract exactly.")

# 2. Subject leakage assertion on persisted metadata
train_subs = set(df_audit[df_audit["split"] == "train"]["subject_id"])
val_subs = set(df_audit[df_audit["split"] == "val"]["subject_id"])
test_subs = set(df_audit[df_audit["split"] == "test"]["subject_id"])

assert len(train_subs & val_subs) == 0, f"Subject overlap Train-Val: {train_subs & val_subs}"
assert len(train_subs & test_subs) == 0, f"Subject overlap Train-Test: {train_subs & test_subs}"
assert len(val_subs & test_subs) == 0, f"Subject overlap Val-Test: {val_subs & test_subs}"
print(f"PASS: Zero subject leakage confirmed across {len(train_subs)} Train, {len(val_subs)} Val, {len(test_subs)} Test subjects.")

# 3. Padded fraction capping assertion
max_pad = df_audit["padded_fraction"].max()
assert max_pad <= 0.20, f"Padded fraction exceeds 20% limit: {max_pad}"
print(f"Repeat-padding contract verified: maximum padded fraction is {max_pad:.4f} (<= 0.20).")

# 4. Feature tensor shape and dtype inspection
print("\nVerifying feature tensor shapes and data types...")
sample_rows = df_audit.sample(min(10, len(df_audit)), random_state=42)
for _, r in sample_rows.iterrows():
    f_path = PROJECT_ROOT / r["feature_path"]
    if f_path.exists():
        arr = np.load(str(f_path))
        assert arr.shape == (499, 768), f"Shape error in {f_path}: got {arr.shape}"
        assert arr.dtype == np.float16, f"Dtype error in {f_path}: got {arr.dtype}"
        assert not np.isnan(arr).any(), f"NaN detected in {f_path}"
        assert not np.isinf(arr).any(), f"Inf detected in {f_path}"

print("Feature tensor verification passed: all inspected tensors have shape (499, 768), float16 dtype, and no NaN/Inf values.")

In [ ]:
# Cell 8: Comprehensive Dataset Distribution Summary
print("=" * 70)
print("             10-SECOND DATASET AUDIT & COHORT SUMMARY")
print("=" * 70)

print(f"Total Extracted 10s Windows: {len(df_audit)}")
print(f"Total Unique Human Subjects: {df_audit['subject_id'].nunique()}")

print("\n1. WINDOWS BY DATASET & SPLIT:")
print(pd.crosstab(df_audit["dataset"], df_audit["split"], margins=True))

print("\n2. WINDOWS BY TASK TYPE & SPLIT:")
print(pd.crosstab(df_audit["task_type"], df_audit["split"], margins=True))

print("\n3. CLASS BALANCE (WINDOWS) BY SPLIT:")
# 0 = Healthy Control, 1 = Parkinson's Disease
print(pd.crosstab(df_audit["split"], df_audit["label"].map({0: "Healthy (0)", 1: "Parkinson's (1)"}), margins=True))

print("\n4. SUBJECT-LEVEL CLASS BALANCE BY SPLIT:")
subj_df = df_audit[["subject_id", "label", "split"]].drop_duplicates()
print(pd.crosstab(subj_df["split"], subj_df["label"].map({0: "Healthy (0)", 1: "Parkinson's (1)"}), margins=True))

print("\n=== PIPELINE READINESS ===")
print("Data preprocessing and feature caching are complete.")
print("Proceed to training the 10-second ConvNeXt V2 + Transformer classifier!")